# Native SAM3 verbose-prompt counting (Kaggle)

Runs the standalone `sam3-verbose-counting/` pipeline end to end: environment setup, gated `sam3.pt` checkpoint download, and text-guided object counting with inline visualizations.

SAM3 takes a **verbose natural-language prompt** (e.g. *"all pairs of black sunglasses displayed on the retail rack"*), returns text-grounded bounding boxes + confidence scores, and the **count is the number of detections above a threshold**.

**Requirements on Kaggle:**
- GPU accelerator enabled (P100/T4 or better; T4 works with fp16 AMP).
- A Kaggle notebook **Secret** named `HF_TOKEN` with an approved token for the gated [`facebook/sam3`](https://huggingface.co/facebook/sam3) model (request access there first). No `hf auth login` needed.

## 1. Setup: clone the repo and install SAM3 runtime deps

Edit `REPO_URL` to point at your actual repository. The cell checks the few SAM3 runtime dependencies that are not guaranteed to ship with the Kaggle base kernel (`timm`, `einops`, `ftfy`, ...) and installs only what is missing into the notebook kernel, then makes `sam3-verbose-counting/` importable.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

# --- 1a. Clone the repository (edit this URL) ------------------------------
REPO_URL = "git@github.com:fez-Ox/pxModel-Object-Counting.git"  # <-- update me
REPO_DIR = Path("/kaggle/working/pxModel-localization")
SAM3_APP = REPO_DIR / "sam3-verbose-counting"

if not SAM3_APP.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repository already present at {REPO_DIR}")

# --- 1b. Install only the SAM3 deps missing from the Kaggle kernel ---------
missing = []
for name in ["torch", "torchvision", "PIL", "numpy", "timm", "einops",
             "ftfy", "regex", "wrapt", "typing_extensions"]:
    try:
        importlib.import_module(name)
    except ImportError:
        missing.append(name)
if "PIL" in missing:
    missing[missing.index("PIL")] = "Pillow"
if missing:
    print("Installing missing deps:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)

# --- 1c. Make the standalone app importable --------------------------------
sys.path.insert(0, str(SAM3_APP))
print("SAM3 app ready:", SAM3_APP)
print("Python:", sys.version.split()[0])

## 2. Download the gated SAM3 checkpoint

The downloader first tries `HF_TOKEN` / `HUGGINGFACE_HUB_TOKEN` env vars, then the Kaggle `HF_TOKEN` notebook Secret, then any `--token` you pass. If the token is not approved for `facebook/sam3`, the checkpoint fetch returns HTTP 401 and you will see a clear message.

In [ ]:
from download_model import download_model, DEFAULT_URL, DEFAULT_OUTPUT

sam3_path = download_model(
    url=DEFAULT_URL,
    output=DEFAULT_OUTPUT,
    force=False,
    timeout=180,
    token=None,  # auto-detects HF_TOKEN / Kaggle secret / --token
)
print("Checkpoint ready:", sam3_path)

## 3. Grab a public sample image

Downloads a COCO image (the classic "cats on a bed" sample used by `main.py`). Replace this with your own file path, folder, or image URL.

In [ ]:
import urllib.request

samples_dir = REPO_DIR / "samples"
samples_dir.mkdir(exist_ok=True)
sample = samples_dir / "coco_cats.jpg"
if not sample.exists():
    urllib.request.urlretrieve(
        "http://images.cocodataset.org/val2017/000000039769.jpg", sample
    )
print("Sample image:", sample)

## 4. Build the persistent SAM3 counter

Loads `sam3.pt` once and reuses the resident model for every image. The model is ~2.5B params, so allow a minute for weights load and the first image's forward pass.

In [ ]:
import torch

from infer import build_counter

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

counter = build_counter(threshold=0.5)  # device auto: cuda when available, else cpu

## 5. Count with a verbose prompt and show the result inline

In [ ]:
from IPython.display import display

from infer import annotate

prompt = "the cats resting on the bed"

result = counter.infer(sample, prompt)
annotated = annotate(sample, prompt, result["boxes"], result["scores"])

print(f"Prompt: {prompt}")
print(f"Count:  {result['count']}")
print(f"Inference time: {result['inference_time_seconds']:.2f}s")
if result["peak_vram_mb"] is not None:
    print(f"Peak VRAM allocated: {result['peak_vram_mb']:.1f} MiB")
    print(f"Peak VRAM reserved:  {result['peak_reserved_vram_mb']:.1f} MiB")
else:
    print("Peak VRAM: unavailable (CPU inference)")

display(annotated)

## 6. Batch: process every image in a folder

The same resident model is reused across all images — only the per-image forward pass runs per call.

In [ ]:
import time

from infer import image_path_to_pil

folder = samples_dir
out_dir = REPO_DIR / "sam3_results"
out_dir.mkdir(exist_ok=True)

images = sorted(p for p in folder.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
print(f"Found {len(images)} image(s) in {folder}")

for image_path in images:
    start = time.perf_counter()
    result = counter.infer(image_path, prompt)
    elapsed = time.perf_counter() - start
    annotated = annotate(image_path, prompt, result["boxes"], result["scores"])
    annotated.save(out_dir / f"{image_path.stem}__annotated.jpg", quality=95)
    print(f"{image_path.name}: count={result['count']}  time={elapsed:.2f}s")
    display(annotated)

print("Saved annotated results to:", out_dir)

## 7. Try your own images and prompts

Swap in your own file path, folder, or URL and a verbose prompt, then re-run the two cells below.

In [ ]:
my_image = "/kaggle/input/my-dataset/my_photo.jpg"  # edit me
my_prompt = "all people wearing blue shirts"       # edit me

my_path = Path(my_image)
if not my_path.exists():
    print(f"File not found: {my_path}. Point `my_image` at a real file or folder.")
else:
    result = counter.infer(my_path, my_prompt)
    annotated = annotate(my_path, my_prompt, result["boxes"], result["scores"])
    print(f"Prompt: {my_prompt}")
    print(f"Count:  {result['count']}")
    display(annotated)

## 8. Cleanup

Releases the model from GPU memory when you are done.

In [ ]:
del counter, annotated
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("GPU cache cleared.")